### Build a modern hybrid transformer: Qwen3.5-0.8B

Implement the **text backbone of Qwen3.5-0.8B** in PyTorch. It is a hybrid:
most layers use **Gated DeltaNet**, a delta-rule recurrence with no quadratic
attention matrix, and every fourth layer uses **full attention** with
partial RoPE and an output gate. For the required work, implement regular
multi-head attention with the same number of query, key, and value heads.
The real checkpoint uses grouped-query attention (GQA), an optional bonus here.

Fill the `TODO` methods in **`qwen.py`** and compare each block against
same-shaped `transformers` modules with copied weights.

The required work is sections 2–9 plus the Hugging Face config converter.
Grouped-query attention (GQA), the chunked recurrence, and Multi-Token
Prediction (MTP) are three independent optional bonuses. The
**week07_efficiency** lecture will explain GQA and many more efficiency
techniques. The small-model tests run on CPU without downloading
checkpoints. Real-weight integration checks are separately opt-in.

Scope: text-only, no KV cache, no padding masks. Forward passes and the
supplied GDN recurrence must support autograd; use `torch.no_grad()` at
inference call sites.

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import transformers
import qwen

#### 1. Config

`Qwen35Config` is given. `layer_types` is derived as:

$$
\text{layer\_types}[i] =
\begin{cases}
\text{"full\_attention"} & \text{if } (i+1) \bmod \text{interval} = 0 \\
\text{"linear\_attention"} & \text{otherwise}
\end{cases}
$$

Same derivation the real `Qwen3_5TextConfig` uses, so a config with matching
`num_hidden_layers`/`full_attention_interval` produces the identical layer
stack -- what lets us steal weights from a real same-shaped model throughout
this notebook. The small config uses smaller dimensions to run on CPU and
sets `num_attention_heads == num_key_value_heads` for regular multi-head
attention. Required tasks vary the number of attention heads while keeping
query/KV counts equal; GQA ratios are covered by the optional bonus. GDN
value/key head ratios remain part of the required work.

In [ ]:
small_config = qwen.make_small_config()
print(small_config)
print("\nlayer_types:", small_config.layer_types)


#### 2. RMSNorm (Qwen's zero-init variant)

Weight init to **zero**; output is `normalized * (1 + weight)`, not
`weight * normalized`:

$$
\hat{x} = \frac{x}{\sqrt{\operatorname{mean}(x^2,\ \text{dim}=-1) + \epsilon}}
$$

$$
\text{RMSNorm}(x) = \hat{x} \odot (1 + w)
$$

Implement `RMSNorm.forward` in `qwen.py`. Compute the normalization and
weight multiplication in float32, then cast the result back to the input
dtype; this matters for float16 and bfloat16.

In [ ]:
norm = qwen.RMSNorm(small_config.hidden_size, eps=small_config.rms_norm_eps)
x = torch.randn(2, 5, small_config.hidden_size)
out = norm(x)
print("output shape:", out.shape, " (weight=0 at init, so this should just be normalized x)")
print("matches plain normalization:", torch.allclose(
    out, x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + norm.eps)
))

#### 3. Gated RMSNorm

Same normalization as RMSNorm, but weight init to **ones**, gated by `silu(gate)`:

$$
\hat{x} = \frac{x}{\sqrt{\operatorname{mean}(x^2,\ \text{dim}=-1) + \epsilon}}
$$

$$
\text{RMSNormGated}(x, g) = w \odot \hat{x} \odot \operatorname{silu}(g)
$$

Implement `RMSNormGated.forward` in `qwen.py`. Match the pinned reference's
rounding boundaries: normalize `x.float()` in float32, cast the normalized
tensor back to `x.dtype`, then multiply by `self.weight` in its parameter
dtype. Next multiply by `silu(gate.float())`, which promotes that product
to float32, and finally cast the output to `x.dtype`. These intermediate
casts matter for float16/bfloat16. Unlike the zero-centered `RMSNorm`
above, gated RMSNorm casts before learned weight scaling.

In [ ]:
gated_norm = qwen.RMSNormGated(small_config.linear_value_head_dim, eps=small_config.rms_norm_eps)
x = torch.randn(2, 5, small_config.linear_value_head_dim)
gate = torch.randn(2, 5, small_config.linear_value_head_dim)
print("output shape:", gated_norm(x, gate).shape)

#### 4. Partial rotary position embedding

Only the first `partial_rotary_factor * head_dim` channels of q/k are
rotated; the rest pass through.

$$
\text{inv\_freq}_i = \theta^{-2i / d_{\text{rot}}},
\qquad i = 0, \dots, d_{\text{rot}}/2 - 1
$$

$$
\text{freqs} = \text{position\_ids} \otimes \text{inv\_freq},
\qquad \text{emb} = [\text{freqs},\ \text{freqs}]
$$

$$
\cos,\ \sin \;=\; \cos(\text{emb}),\ \sin(\text{emb})
$$

$$
\text{rotate\_half}([x_1, x_2]) = [-x_2,\ x_1]
$$

$$
x_{\text{rot}}' = x_{\text{rot}} \odot \cos + \text{rotate\_half}(x_{\text{rot}}) \odot \sin
$$

The rotary constructor is given. Implement `Qwen35RotaryEmbedding.forward`, `rotate_half`, and
`apply_rotary_pos_emb` in `qwen.py`. See
`tests/test_qwen_public.py::TestRotaryEmbedding` for exact shapes.

In [ ]:
rope = qwen.Qwen35RotaryEmbedding(small_config)
seq_len = 10
position_ids = torch.arange(seq_len).unsqueeze(0)
dummy_hidden = torch.randn(1, seq_len, small_config.hidden_size)
cos, sin = rope(dummy_hidden, position_ids)
print("cos/sin shape:", cos.shape, "(batch, seq_len, rotary_dim) -- rotary_dim =", cos.shape[-1],
      "= head_dim * partial_rotary_factor =", small_config.head_dim, "*", small_config.partial_rotary_factor)

q = torch.randn(1, 4, seq_len, small_config.head_dim)
k = torch.randn(1, 4, seq_len, small_config.head_dim)
q_rot, k_rot = qwen.apply_rotary_pos_emb(q, k, cos, sin)
rotary_dim = cos.shape[-1]
print("pass-through channels untouched:", torch.equal(q_rot[..., rotary_dim:], q[..., rotary_dim:]))

#### 5. Full attention: regular multi-head attention + per-head QK-norm + partial RoPE + output gate

Implement regular multi-head attention with one query, key, and value head
per attention head (`num_attention_heads == num_key_value_heads`). Add
per-head RMSNorm on q/k, partial RoPE, and a sigmoid output gate, all
vectorized (no loop over heads):

$$
Q = \text{q\_norm}(\text{q\_proj}(x)), \qquad
K = \text{k\_norm}(\text{k\_proj}(x)), \qquad
V = \text{v\_proj}(x)
$$

$$
\text{scores} = \frac{Q K^\top}{\sqrt{d_{\text{head}}}} + \text{mask}
$$

$$
\text{attn} = \operatorname{softmax}(\text{scores})\, V
$$

$$
\text{out} = \text{o\_proj}\big(\text{attn} \odot \sigma(\text{gate})\big)
$$

$Q$, $K$, and $V$ have the same number of heads, so each query head uses
its corresponding key/value head directly. The gate is packed alongside
the query in one projection.

The constructor and QKV/gate projection helper are given. Implement
`GatedAttention.forward` in `qwen.py` with batched matrix multiplication
and softmax: no Python loop over heads and no
`torch.nn.functional.scaled_dot_product_attention` (SDPA). Honor the
additive attention mask. See `tests/test_qwen_public.py::TestGatedAttention`
for exact projections/shapes. If `attention_mask` is `None`, create a
causal mask; otherwise use the supplied additive mask.

The supplied dispatch sends configs with fewer key/value heads to the
optional `GatedAttention.forward_gqa` method. Leave that TODO unfinished
for the required work. We will explain **GQA and many more efficiency
techniques in week07_efficiency**; the bonus below is available if you
want to explore it now.

The supplied `naive_looped_gated_attention` uses the same explicit
matmul/softmax operations one head at a time. Use it to sanity-check
correctness and inspect timing. A fixed wall-clock speedup is not a grading
requirement: dispatch costs and CPU/GPU backends vary by machine.

In [ ]:
layer_idx = small_config.layer_types.index("full_attention")
attn = qwen.GatedAttention(small_config, layer_idx)
attn.eval()

batch, seq_len = 2, 12
hidden_states = torch.randn(batch, seq_len, small_config.hidden_size)
mask = qwen.build_causal_mask(seq_len, hidden_states.dtype, hidden_states.device)
position_embeddings = qwen.Qwen35RotaryEmbedding(small_config)(
    hidden_states, torch.arange(seq_len).unsqueeze(0).expand(batch, -1)
)

with torch.no_grad():
    out_fast = attn(hidden_states, position_embeddings, mask)
    out_naive = qwen.naive_looped_gated_attention(attn, hidden_states, position_embeddings, mask)

print("output shape:", out_fast.shape)
print("matches naive per-head loop:", torch.allclose(out_fast, out_naive, atol=1e-4))

#### 6. SwiGLU MLP

Standard SwiGLU:

$$
\text{MLP}(x) = \text{down\_proj}\Big(
\operatorname{silu}\big(\text{gate\_proj}(x)\big) \odot \text{up\_proj}(x)
\Big)
$$

No surprises here -- implement `MLP.forward` in `qwen.py`.

In [ ]:
mlp = qwen.MLP(small_config)
x = torch.randn(2, 5, small_config.hidden_size)
print("output shape:", mlp(x).shape)

#### 7. Gated DeltaNet (linear attention)

Maintains a fixed-size recurrent state $S \in \mathbb{R}^{d_k \times d_v}$
per head, updated by a **delta rule**: compare state against the true
value, write the difference scaled by $\beta$, with per-head decay $g$:

$$
S_t \leftarrow S_{t-1} \cdot \exp(g_t)
$$

$$
\text{kv\_mem}_t = S_t^\top k_t
$$

$$
\delta_t = \big(v_t - \text{kv\_mem}_t\big) \cdot \beta_t
$$

$$
S_t \leftarrow S_t + k_t \otimes \delta_t
$$

$$
\text{out}_t = S_t^\top q_t
$$

$$
\beta_t = \sigma\big(\text{in\_proj\_b}(x_t)\big), \qquad
g_t = -\exp(A_{\log}) \cdot \operatorname{softplus}\big(\text{in\_proj\_a}(x_t) + \text{dt\_bias}\big)
$$

The constructor and recurrence are given. Implement **`GatedDeltaNet.forward`**.
For input shape `(batch, seq_len, hidden_size)`, let
`K = num_k_heads * head_k_dim` and `V = num_v_heads * head_v_dim`:

1. Apply `in_proj_qkv`, giving `(batch, seq_len, 2*K + V)`. Transpose to
   `(batch, 2*K + V, seq_len)` for the supplied depthwise `conv1d`. Its
   symmetric padding produces extra timesteps: keep only the **first
   `seq_len` outputs** so each output depends only on current/past input.
   Apply SiLU and transpose back.
2. Split the final dimension into widths **`K`, `K`, `V`**, in query, key,
   value order. Reshape query/key to
   `(batch, seq_len, num_k_heads, head_k_dim)` and value to
   `(batch, seq_len, num_v_heads, head_v_dim)`.
3. When `num_v_heads > num_k_heads`, repeat each query/key head
   `num_v_heads // num_k_heads` times consecutively. All three tensors
   passed to the recurrence must have `num_v_heads` heads. Their key and
   value dimensions need not be equal.
4. From the **original hidden states**, compute `beta = sigmoid(in_proj_b(x))`
   and `g` by the decay formula above. Both have shape
   `(batch, seq_len, num_v_heads)`. Compute decay in float32, including
   `exp(A_log.float())` and `softplus(in_proj_a(x).float() + dt_bias.float())`.
   Both `A_log` and `dt_bias` matter, and neither is a constant to omit.
5. Call the supplied `gated_delta_rule_recurrent(query, key, value, g, beta)`.
   It already L2-normalizes query/key, scales query by `1/sqrt(head_k_dim)`,
   and returns `(batch, seq_len, num_v_heads, head_v_dim)`. Do not apply
   these normalization/scaling steps a second time.
6. Compute `z = in_proj_z(x)` from the original hidden states and reshape
   it to that same output shape. Apply `self.norm(core_output, z)` per
   value head, using the supplied `RMSNormGated` module. Flatten the two
   head dimensions, then apply `out_proj` to return
   `(batch, seq_len, hidden_size)`.

Preserve the computation graph through convolution, projections, decay,
recurrence, and gated normalization. The supplied sequential recurrence
supports backward; do not detach its inputs or wrap this forward in
`torch.no_grad()`. See `tests/test_qwen_public.py::TestGatedDeltaNet`
for module names and examples.

The chunked recurrence is an independent optional bonus at the end.

In [ ]:
batch, seq_len = 2, 37
heads, k_dim, v_dim = small_config.linear_num_value_heads, small_config.linear_key_head_dim, small_config.linear_value_head_dim
query = torch.randn(batch, seq_len, heads, k_dim)
key = torch.randn(batch, seq_len, heads, k_dim)
value = torch.randn(batch, seq_len, heads, v_dim)
g = -torch.rand(batch, seq_len, heads)
beta = torch.rand(batch, seq_len, heads)

out_recurrent = qwen.gated_delta_rule_recurrent(query, key, value, g, beta)
print("output shape:", out_recurrent.shape)

In [ ]:
layer_idx = small_config.layer_types.index("linear_attention")
gdn = qwen.GatedDeltaNet(small_config, layer_idx)
gdn.eval()
hidden_states = torch.randn(2, 19, small_config.hidden_size)
with torch.no_grad():
    print("GatedDeltaNet output shape:", gdn(hidden_states).shape)

#### 8. Decoder layer: assembling attention/GDN + MLP into a residual block

Pre-LN residual block, dispatching to whichever attention module this layer got:

$$
h \leftarrow h + \text{Attn}\big(\text{input\_layernorm}(h)\big)
$$

$$
h \leftarrow h + \text{MLP}\big(\text{post\_attention\_layernorm}(h)\big)
$$

The constructor is given. Implement `Qwen35DecoderLayer.forward` in `qwen.py`. See
`tests/test_qwen_public.py::TestDecoderLayer` for exact attribute names.

In [ ]:
layer_idx = small_config.layer_types.index("linear_attention")
layer = qwen.Qwen35DecoderLayer(small_config, layer_idx)
layer.eval()

batch, seq_len = 2, 12
hidden_states = torch.randn(batch, seq_len, small_config.hidden_size)
mask = qwen.build_causal_mask(seq_len, hidden_states.dtype, hidden_states.device)
position_embeddings = qwen.Qwen35RotaryEmbedding(small_config)(
    hidden_states, torch.arange(seq_len).unsqueeze(0).expand(batch, -1)
)
with torch.no_grad():
    out = layer(hidden_states, position_embeddings, mask)
print("output shape:", out.shape, " block_type:", layer.block_type)

#### 9. The full model: `Qwen35Model` and `Qwen35ForCausalLM`

`Qwen35Model` stacks decoder layers between an embedding table and a final
`RMSNorm`. `Qwen35ForCausalLM` adds the LM head:

$$
\text{logits} = h\, W_{\text{lm\_head}}^\top
$$

$$
\mathcal{L} = \operatorname{CrossEntropy}(\text{logits},\ \text{labels})
$$

`lm_head` is a real `nn.Linear`, weight-aliased to `embed_tokens.weight`
when `tie_word_embeddings=True` -- numerically identical to the tied
matmul. Untied checkpoints (see "scaling to other sizes" below) work the
same way.

Implement `Qwen35Model.forward` and `Qwen35ForCausalLM.forward` in `qwen.py`.
The model embeds input IDs, builds positions `0..seq_len-1` and an additive
causal mask on the input device, applies the decoder stack, then the final
norm. Return a hidden-state tensor, not a `transformers` output object.
The LM wrapper returns `(logits, loss)`, with `loss=None` when labels are absent.

**Labels are already aligned with logits.** This assignment's wrapper does
not shift them internally. For next-token training, pass
`input_ids=tokens[:, :-1]` and `labels=tokens[:, 1:]`; apply cross entropy
at matching positions. Labels may contain `-100` to ignore an individual
loss term (PyTorch's default `ignore_index`). This loss mask does not add
padding-mask support to attention.

In [ ]:
model = qwen.Qwen35ForCausalLM(small_config)
model.eval()
tokens = torch.randint(0, small_config.vocab_size, (2, 17))
input_ids, labels = tokens[:, :-1], tokens[:, 1:]
with torch.no_grad():
    logits, loss = model(input_ids, labels=labels)
print("logits shape:", logits.shape)
print("loss:", loss.item())

#### Real weights: load the actual Qwen3.5-0.8B checkpoint and generate text

Downloads the real `Qwen/Qwen3.5-0.8B` checkpoint and loads it into your
`Qwen35ForCausalLM`, then compares greedy decoding against real
`transformers` on the same weights/tokens -- a direct end-to-end check
beyond the unit tests.

This opt-in check downloads roughly 1.6–1.8 GB into `HF_HOME`
(default `.cache/huggingface`). Set
`RUN_REAL_WEIGHTS_DEMO=True` after completing the config converter below
**and the GQA bonus**. The checkpoint has fewer key/value heads than query
heads, so completing only the required regular-attention TODO is not enough
for this demonstration. The matching integration tests live in
`tests/test_qwen_real_weights.py`.

In [ ]:
RUN_REAL_WEIGHTS_DEMO = False

if RUN_REAL_WEIGHTS_DEMO:
    from huggingface_hub import snapshot_download
    from safetensors.torch import load_file
    import glob, os

    hf_config = transformers.AutoConfig.from_pretrained("Qwen/Qwen3.5-0.8B")

    real_config = qwen.qwen35_config_from_hf(hf_config.to_dict())
    real_model = qwen.Qwen35ForCausalLM(real_config)

    ckpt_dir = snapshot_download("Qwen/Qwen3.5-0.8B")
    state_dict = {}
    for path in sorted(glob.glob(os.path.join(ckpt_dir, "*.safetensors"))):
        state_dict.update(load_file(path))

    prefix = "model.language_model."
    clean_state_dict = {
        key[len(prefix):]: value for key, value in state_dict.items()
        if key.startswith(prefix) and not key.startswith(prefix + "mtp.")
    }
    real_model.model.load_state_dict(clean_state_dict, strict=True)

    tokenizer = transformers.AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B")
    real_model.eval()
    tokens = tokenizer.encode("The Fermi paradox ")
    for _ in range(60):
        with torch.no_grad():
            logits, _ = real_model(torch.as_tensor([tokens]))
        p_next = torch.softmax(logits[0, -1], dim=-1).numpy()
        tokens.append(int(p_next.argmax()))
    print(tokenizer.decode(tokens))
else:
    print("Skipped (RUN_REAL_WEIGHTS_DEMO=False).")

In [ ]:
from transformers.models.qwen3_5 import modeling_qwen3_5 as ref

if RUN_REAL_WEIGHTS_DEMO:
    real_text_model = ref.Qwen3_5TextModel(hf_config.get_text_config())
    real_text_model.load_state_dict(clean_state_dict, strict=True)
    real_text_model.eval()

    ref_tokens = tokenizer.encode("The Fermi paradox ")
    for _ in range(60):
        with torch.no_grad():
            hidden = real_text_model(torch.as_tensor([ref_tokens])).last_hidden_state
            logits = hidden @ real_text_model.embed_tokens.weight.T
        p_next = torch.softmax(logits[0, -1], dim=-1).numpy()
        ref_tokens.append(int(p_next.argmax()))
    print(tokenizer.decode(ref_tokens))
    assert ref_tokens == tokens, "Greedy token sequences differ from the reference"
else:
    print("Skipped (RUN_REAL_WEIGHTS_DEMO=False).")

#### Scaling to other Qwen3.5 sizes

Almost nothing here is 0.8B-specific: the dense family (0.8B/4B/9B/27B)
shares `full_attention_interval`, `head_dim`, `rope_theta`,
`partial_rotary_factor`; only per-size dimensions and the linear-head ratio
change (already handled by `repeat_interleave` in `GatedDeltaNet.forward`).
These checkpoints also use GQA, so running them requires the GQA bonus;
converting their configs is still part of the required work. Preserve their
`num_key_value_heads` rather than replacing it with `num_attention_heads`.

One real gap: 0.8B/4B tie word embeddings, **9B/27B don't** -- already
handled by `Qwen35ForCausalLM`'s real `nn.Linear` `lm_head`.

(Qwen3.5-35B-A3B is a different, MoE architecture -- out of scope.)

Implement `qwen35_config_from_hf` in `qwen.py`, mapping a real model's
config.json into a `Qwen35Config`:

- https://huggingface.co/Qwen/Qwen3.5-0.8B/blob/main/config.json
- https://huggingface.co/Qwen/Qwen3.5-4B/blob/main/config.json
- https://huggingface.co/Qwen/Qwen3.5-9B/blob/main/config.json
- https://huggingface.co/Qwen/Qwen3.5-27B/blob/main/config.json

Copy all represented architecture fields, including `linear_conv_kernel_dim`,
`linear_key_head_dim`, `linear_value_head_dim`, both GDN head counts,
normalization epsilon, activation, attention bias, context length, and any
explicit `layer_types`. Leaving the small-config head dimensions at 16
instead of the checkpoint's 128 gives an incompatible model. Also map
`mtp_num_hidden_layers` to `mtp_num_layers`.

Accept both a full dictionary containing `text_config` and a text-only
dictionary. For full dictionaries, a top-level `tie_word_embeddings` takes
precedence; otherwise use the text value, then the dataclass default. Read
RoPE values from `rope_parameters` (or the legacy `rope_scaling` dictionary),
with flat values/defaults as fallbacks. See the function's docstring in
`qwen.py` for the remaining naming hints.

In [ ]:
# A literal, offline copy of Qwen3.5-4B's config.json (the parts we need) --
# pasted directly rather than downloaded, so this cell needs no network access.
qwen35_4b_config = {
    "tie_word_embeddings": True,
    "text_config": {
        "vocab_size": 248320, "hidden_size": 2560, "intermediate_size": 9216,
        "num_hidden_layers": 32, "num_attention_heads": 16, "num_key_value_heads": 4,
        "head_dim": 256, "attention_bias": False,
        "linear_conv_kernel_dim": 4, "linear_key_head_dim": 128, "linear_value_head_dim": 128,
        "linear_num_key_heads": 16, "linear_num_value_heads": 32,
        "max_position_embeddings": 262144, "rms_norm_eps": 1e-06, "hidden_act": "silu",
        "full_attention_interval": 4, "mtp_num_hidden_layers": 1,
        "tie_word_embeddings": True,
        "rope_parameters": {"rope_theta": 10000000, "partial_rotary_factor": 0.25},
    },
}

config_4b = qwen.qwen35_config_from_hf(qwen35_4b_config)
print(config_4b)
print("\nnum layer_types:", len(config_4b.layer_types), " full_attention count:",
      config_4b.layer_types.count("full_attention"))

#### Bonus (optional): grouped-query attention (GQA)

Implement `GatedAttention.forward_gqa` in `qwen.py`. It extends the required
regular attention to configs where `num_key_value_heads` is smaller than
`num_attention_heads` and divides it exactly. The supplied `forward`
dispatch calls this method for those configs; the default small model uses
regular attention and does not need this bonus.

The projection helper returns query heads with shape `(batch, Hq, seq_len,
head_dim)` and key/value heads with shape `(batch, Hkv, seq_len, head_dim)`.
Let `groups = Hq // Hkv`. Query head `h` must use key/value head
`h // groups`: for four query heads and two KV heads, the mapping is
`[0, 0, 1, 1]`. Repeat each KV head **consecutively** along the head axis
(or use an equivalent grouped batched computation), then apply the same
scaled scores, additive/causal mask, softmax, sigmoid gate, and output
projection as regular attention. Preserve gradients and vectorize across
heads; do not call SDPA or loop over individual heads.

The **week07_efficiency** lecture will explain GQA and many more efficiency
techniques. This optional extension enables the real-checkpoint example
above, because Qwen3.5-0.8B uses GQA. It is independent of the chunked
recurrence and MTP bonuses. Set `RUN_GQA_BONUS=True` below when ready.


In [ ]:
RUN_GQA_BONUS = False

if RUN_GQA_BONUS:
    gqa_config = qwen.make_small_config(num_attention_heads=4, num_key_value_heads=2)
    layer_idx = gqa_config.layer_types.index("full_attention")
    gqa = qwen.GatedAttention(gqa_config, layer_idx).eval()
    batch, seq_len = 2, 12
    hidden_states = torch.randn(batch, seq_len, gqa_config.hidden_size)
    mask = qwen.build_causal_mask(seq_len, hidden_states.dtype, hidden_states.device)
    position_embeddings = qwen.Qwen35RotaryEmbedding(gqa_config)(
        hidden_states, torch.arange(seq_len).unsqueeze(0).expand(batch, -1)
    )
    with torch.no_grad():
        out_gqa = gqa(hidden_states, position_embeddings, mask)
        out_naive = qwen.naive_looped_gated_attention(
            gqa, hidden_states, position_embeddings, mask
        )
    print("output shape:", out_gqa.shape)
    print("matches naive per-head loop:", torch.allclose(out_gqa, out_naive, atol=1e-4))
else:
    print("Skipped optional GQA bonus.")


#### Bonus (optional): efficient chunked delta rule

`gated_delta_rule_recurrent` is correct but loops over `seq_len` in Python
-- slow on GPU at long sequences. Real `transformers` restructures the same
math into chunks (the "WY representation" trick): correct for intra-chunk
interactions with a handful of chunk-sized matmuls, then scan across chunks
carrying a small recurrent state, instead of scanning every timestep.

Implement `gated_delta_rule_chunked` in `qwen.py` so it computes the same
result as `gated_delta_rule_recurrent` within floating-point tolerance.
Only the recurrence between chunks may scan the sequence: use
`O(seq_len / chunk_size)` state updates, with batched intra-chunk work
(a correction loop bounded by `chunk_size` is allowed). This is the hardest block in the assignment -- work out the
WY-representation algorithm yourself (real `transformers`'
`torch_chunk_gated_delta_rule` is one place to look) and cross-check against
`gated_delta_rule_recurrent` at a small, non-chunk-size-multiple `seq_len`.

This function is not wired into `GatedDeltaNet.forward`. Public examples
live in `tests/test_qwen_bonus.py`. Merely calling
or copying the timestep recurrence does not satisfy the bonus, even though
it produces correct outputs and linear timing growth. Handle sequence lengths that do not divide the chunk size, carry state
across chunk boundaries, and batch intra-chunk computation; runtime
scaling alone is not proof of chunking. Set `RUN_CHUNKED_BONUS=True` below only when ready.

In [ ]:
RUN_CHUNKED_BONUS = False

if RUN_CHUNKED_BONUS:
    batch, seq_len = 2, 37  # deliberately NOT a multiple of chunk_size=64
    heads, k_dim, v_dim = small_config.linear_num_value_heads, small_config.linear_key_head_dim, small_config.linear_value_head_dim
    query = torch.randn(batch, seq_len, heads, k_dim)
    key = torch.randn(batch, seq_len, heads, k_dim)
    value = torch.randn(batch, seq_len, heads, v_dim)
    g = -torch.rand(batch, seq_len, heads)
    beta = torch.rand(batch, seq_len, heads)

    out_chunked = qwen.gated_delta_rule_chunked(query, key, value, g, beta)
    out_recurrent = qwen.gated_delta_rule_recurrent(query, key, value, g, beta)
    print("output shape:", out_chunked.shape)
    print("matches the recurrent reference:", torch.allclose(out_chunked, out_recurrent, atol=1e-3))
else:
    print("Skipped optional chunked bonus.")

#### Bonus (optional): Multi-Token Prediction (MTP) head

Implement the educational `Qwen35MTPHead.forward` in `qwen.py`. For a
sequence of length `L >= 3`, output position `t` predicts token `t+2` using
the backbone state at `t` and the known embedding of token `t+1`.

1. Take `hidden_states[:, :-2]` and `embed_tokens(input_ids[:, 1:-1])`.
   Normalize them separately with `pre_fc_norm_hidden` and
   `pre_fc_norm_embedding`.
2. Concatenate **hidden state first, embedding second** on the last axis,
   then apply `fc` to get `(batch, L-2, hidden_size)` features.
3. Build a causal mask of length `L-2` and RoPE for absolute positions
   **`1..L-2`**, corresponding to the embedded next tokens. Run every layer
   in `self.layers`; each is a full-attention decoder layer.
4. Apply the final `norm`, then multiply by `embed_tokens.weight.T` to
   produce logits of shape `(batch, L-2, vocab_size)`. Return logits only.
   Targets are `input_ids[:, 2:]`; do not read those target embeddings as
   inputs. For `L < 3`, raise `ValueError` because there is no prediction target.

This simplified bonus always uses the embedding matrix as the output
classifier, including when the backbone has an untied LM head. It is not a
checkpoint-compatible implementation of Qwen's full MTP/speculative-decoding
system. The constructor is given and supports `mtp_num_layers` decoder
layers; preserve gradients through both feature streams and all layers.

Preserve alignment, causality, the specified computation, and gradients
in addition to producing the required shapes. Set `RUN_MTP_BONUS=True` below when
ready. This bonus uses the equal-head `small_config` and is independent
of both the GQA and chunked bonuses.

In [ ]:
RUN_MTP_BONUS = False

if RUN_MTP_BONUS:
    mtp = qwen.Qwen35MTPHead(small_config)
    model.eval(); mtp.eval()

    batch, seq_len = 2, 16
    input_ids = torch.randint(0, small_config.vocab_size, (batch, seq_len))
    with torch.no_grad():
        hidden_states = model.model(input_ids)
        mtp_logits = mtp(hidden_states, input_ids, model.model.embed_tokens)

    print("mtp logits shape:", mtp_logits.shape, " (expected: (batch, seq_len - 2, vocab_size))")
else:
    print("Skipped optional MTP bonus.")